In [4]:
# Scope-aware hybrid Recall@30 diagnostic
# Run this cell from the repository root or notebooks/.

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from src.embeddings.embed_chunks import MODEL_CONFIGS
from src.scripts.evaluate_bge_bm25_fusion import (
    DEFAULT_CHUNKS_DIRECTORY, DEFAULT_EMBEDDINGS_DIRECTORY, DEFAULT_TEST_QUERIES_PATH,
    MAX_K, build_bm25_index, evaluate_query, load_corpus, load_test_queries,
)
from src.scripts.evaluate_scope_aware_hybrid_retrieval import (
    DEFAULT_RRF_K, detect_scope, hybrid_retrieve, scope_aware_hybrid_retrieve,
    dense_candidate_indices, bm25_candidate_indices,
)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'evaluation' / 'scope_aware_recall_diagnostics' / 'bgebase'
existing_versions = [int(path.name[1:]) for path in OUTPUT_ROOT.glob('v*') if path.is_dir() and path.name[1:].isdigit()]
OUTPUT_DIR = OUTPUT_ROOT / f'v{max(existing_versions, default=0) + 1}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

MODEL_NAME = 'bgebase'
MODEL_CONFIG = MODEL_CONFIGS[MODEL_NAME]
TOP_K = 30
RRF_K = DEFAULT_RRF_K
EXPERIMENTS = [
    {'name': 'scope_aware_candidate_30', 'mode': 'scope_aware', 'candidate_k': 30},
    {'name': 'oracle_company_candidate_30', 'mode': 'oracle_company', 'candidate_k': 30},
    {'name': 'oracle_company_candidate_50', 'mode': 'oracle_company', 'candidate_k': 50},
    {'name': 'oracle_company_candidate_100', 'mode': 'oracle_company', 'candidate_k': 100},
    {'name': 'oracle_company_candidate_200', 'mode': 'oracle_company', 'candidate_k': 200},
]

all_embeddings, all_chunks = load_corpus(
    PROJECT_ROOT / DEFAULT_EMBEDDINGS_DIRECTORY, PROJECT_ROOT / DEFAULT_CHUNKS_DIRECTORY, MODEL_NAME
)
normalized_embeddings = all_embeddings / np.clip(np.linalg.norm(all_embeddings, axis=1, keepdims=True), 1e-12, None)
bm25_retriever = build_bm25_index(all_chunks)
model = SentenceTransformer(MODEL_CONFIG['repository'])
test_cases = load_test_queries(PROJECT_ROOT / DEFAULT_TEST_QUERIES_PATH)
chunks_by_id = {chunk['chunk_id']: chunk for chunk in all_chunks}
indices_by_ticker = {
    ticker: np.asarray([index for index, chunk in enumerate(all_chunks) if chunk.get('ticker') == ticker])
    for ticker in {chunk.get('ticker') for chunk in all_chunks}
}
print(f'Loaded {len(test_cases)} queries and {len(all_chunks)} chunks. Output: {OUTPUT_DIR}')

def exact_within_company_ranks(query, ticker):
    """Full within-company rankings; used only to diagnose misses."""
    allowed = indices_by_ticker[ticker]
    dense_indices, _ = dense_candidate_indices(
        query, model, MODEL_CONFIG['query_prefix'], normalized_embeddings, len(allowed), allowed
    )
    bm25_indices, _ = bm25_candidate_indices(
        query, bm25_retriever, len(all_chunks), len(allowed), allowed
    )
    return (
        {all_chunks[index]['chunk_id']: rank for rank, index in enumerate(dense_indices, start=1)},
        {all_chunks[index]['chunk_id']: rank for rank, index in enumerate(bm25_indices, start=1)},
    )

def current_scope_aware_pool(query, candidate_k):
    """Return the same candidate pool/ranks used by the current scope-aware path."""
    scope, companies = detect_scope(query)
    if scope == 'single_company':
        raw = hybrid_retrieve(query, model, MODEL_CONFIG['query_prefix'], normalized_embeddings, bm25_retriever, all_chunks, RRF_K, candidate_k, {companies[0]})
        return raw, raw[:TOP_K], scope, companies
    if scope == 'explicit_subset':
        raw = hybrid_retrieve(query, model, MODEL_CONFIG['query_prefix'], normalized_embeddings, bm25_retriever, all_chunks, RRF_K, candidate_k, set(companies))
        return raw, raw[:TOP_K], scope, companies
    if scope == 'global':
        raw = hybrid_retrieve(query, model, MODEL_CONFIG['query_prefix'], normalized_embeddings, bm25_retriever, all_chunks, RRF_K, candidate_k)
        return raw, raw[:TOP_K], scope, companies
    # Anchored-global is evaluated through the production helper. Its appended anchor
    # candidates are already deduplicated; global RRF rank is retained when available.
    final, scope, companies = scope_aware_hybrid_retrieve(
        query, model, MODEL_CONFIG['query_prefix'], normalized_embeddings, bm25_retriever,
        all_chunks, RRF_K, candidate_k, TOP_K, 3,
    )
    raw = hybrid_retrieve(query, model, MODEL_CONFIG['query_prefix'], normalized_embeddings, bm25_retriever, all_chunks, RRF_K, candidate_k)
    return raw, final[:TOP_K], scope, companies

def run_variant(case, experiment):
    query, gold_ticker = case['question'], case['ticker']
    if experiment['mode'] == 'scope_aware':
        raw_pool, final_results, scope, companies = current_scope_aware_pool(query, experiment['candidate_k'])
    else:
        raw_pool = hybrid_retrieve(
            query, model, MODEL_CONFIG['query_prefix'], normalized_embeddings, bm25_retriever,
            all_chunks, RRF_K, experiment['candidate_k'], {gold_ticker},
        )
        final_results, scope, companies = raw_pool[:TOP_K], 'oracle_company', [gold_ticker]
    for rank, result in enumerate(raw_pool, start=1):
        result['rrf_rank_in_candidate_pool'] = rank
    return raw_pool, final_results, scope, companies

def metric_summary(frame):
    return {
        metric: float(frame[metric].mean())
        for metric in ('recall@30', 'hit@30', 'complete@30', 'mrr@30')
    }

all_rows, missed_rows, comparison_rows = [], [], []
for experiment in EXPERIMENTS:
    rows = []
    experiment_start = time.perf_counter()
    for number, case in enumerate(test_cases, start=1):
        raw_pool, final_results, scope, detected_companies = run_variant(case, experiment)
        final_ids = [result['chunk_id'] for result in final_results]
        row = evaluate_query(case, final_results, chunks_by_id)
        row.update({
            'experiment': experiment['name'], 'candidate_k': experiment['candidate_k'],
            'scope': scope, 'detected_companies': detected_companies,
            'retrieved_chunk_ids': final_ids, 'candidate_pool_chunk_ids': [result['chunk_id'] for result in raw_pool],
        })
        rows.append(row)
        rrf_ranks = {result['chunk_id']: result['rrf_rank_in_candidate_pool'] for result in raw_pool}
        dense_ranks, bm25_ranks = exact_within_company_ranks(case['question'], case['ticker'])
        for gold_chunk_id in case['expected_chunk_ids']:
            if gold_chunk_id in final_ids:
                continue
            dense_rank, bm25_rank = dense_ranks.get(gold_chunk_id), bm25_ranks.get(gold_chunk_id)
            entered_pool = gold_chunk_id in rrf_ranks
            missed_rows.append({
                'experiment': experiment['name'], 'candidate_k': experiment['candidate_k'],
                'question': case['question'], 'ticker': case['ticker'], 'question_type': case['question_type'],
                'gold_chunk_id': gold_chunk_id, 'within_company_dense_rank': dense_rank,
                'within_company_bm25_rank': bm25_rank,
                'rrf_rank_in_candidate_pool': rrf_ranks.get(gold_chunk_id),
                'entered_rrf_candidate_pool': entered_pool,
                'excluded_by_dense_candidate_k': dense_rank is not None and dense_rank > experiment['candidate_k'],
                'excluded_by_bm25_candidate_k': bm25_rank is not None and bm25_rank > experiment['candidate_k'],
                'excluded_because_candidate_k_too_small': not entered_pool and (
                    (dense_rank is not None and dense_rank > experiment['candidate_k']) and
                    (bm25_rank is not None and bm25_rank > experiment['candidate_k'])
                ),
                'scope': scope, 'detected_companies': detected_companies,
            })
        if number % 25 == 0 or number == len(test_cases):
            print(f"{experiment['name']}: {number}/{len(test_cases)}")
    frame = pd.DataFrame(rows)
    all_rows.extend(rows)
    summary = metric_summary(frame)
    summary.update({
        'experiment': experiment['name'], 'candidate_k': experiment['candidate_k'],
        'mode': experiment['mode'], 'seconds': time.perf_counter() - experiment_start,
        'missed_gold_chunks': sum(len(case['expected_chunk_ids']) for case in test_cases) - sum(
            len(set(case['expected_chunk_ids']) & set(ids)) for case, ids in zip(test_cases, frame['retrieved_chunk_ids'])
        ),
    })
    comparison_rows.append(summary)
    print(experiment['name'], {key: round(value, 4) for key, value in summary.items() if key.endswith('@30')})

comparison = pd.DataFrame(comparison_rows)[
    ['experiment', 'mode', 'candidate_k', 'recall@30', 'hit@30', 'complete@30', 'mrr@30', 'missed_gold_chunks', 'seconds']
]
comparison = comparison.sort_values(['mode', 'candidate_k'], kind='stable')
print('\nComparison')
print(comparison.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

pd.DataFrame(all_rows).to_json(OUTPUT_DIR / 'evaluation.jsonl', orient='records', lines=True, force_ascii=False)
pd.DataFrame(missed_rows).to_json(OUTPUT_DIR / 'missed_gold_chunks.jsonl', orient='records', lines=True, force_ascii=False)
comparison.to_csv(OUTPUT_DIR / 'comparison.csv', index=False)
with (OUTPUT_DIR / 'summary.json').open('w', encoding='utf-8') as file:
    json.dump({
        'top_k': TOP_K, 'rrf_k': RRF_K, 'experiments': comparison_rows,
        'diagnostic_fields': [
            'within_company_dense_rank', 'within_company_bm25_rank',
            'rrf_rank_in_candidate_pool', 'excluded_because_candidate_k_too_small',
        ],
    }, file, indent=2)
print(f'\nSaved evaluation.jsonl, missed_gold_chunks.jsonl, comparison.csv, and summary.json to {OUTPUT_DIR}')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3610.29it/s]        


Loaded 300 queries and 4115 chunks. Output: /home/veselin/Documents/Programiranje/edgar-insight-rag/data/evaluation/scope_aware_recall_diagnostics/bgebase/v3


scope_aware_candidate_30: 25/300


scope_aware_candidate_30: 50/300


scope_aware_candidate_30: 75/300


...

oracle_company_candidate_200: 200/300


oracle_company_candidate_200: 225/300


oracle_company_candidate_200: 250/300


oracle_company_candidate_200: 275/300


oracle_company_candidate_200: 300/300
oracle_company_candidate_200 {'recall@30': 0.9539, 'hit@30': 0.98, 'complete@30': 0.93, 'mrr@30': 0.6411}

Comparison
                  experiment           mode  candidate_k  recall@30  hit@30  complete@30  mrr@30  missed_gold_chunks  seconds
 oracle_company_candidate_30 oracle_company           30     0.9539  0.9800       0.9267  0.6450                  25 125.4981
 oracle_company_candidate_50 oracle_company           50     0.9556  0.9833       0.9300  0.6406                  25 240.3862
oracle_company_candidate_100 oracle_company          100     0.9383  0.9767       0.9000  0.6409                  34 371.6304
oracle_company_candidate_200 oracle_company          200     0.9539  0.9800       0.9300  0.6411                  25 124.9946
    scope_aware_candidate_30    scope_aware           30     0.9506  0.9767       0.9233  0.6195                  26 118.4810

Saved evaluation.jsonl, missed_gold_chunks.jsonl, comparison.csv, and summary.json to /

In [5]:
# Inspect the remaining oracle-company misses after the diagnostic cell has run.
# Change this only when inspecting a different oracle candidate depth.
INSPECTION_EXPERIMENT = 'oracle_company_candidate_200'
TOP_RETRIEVED_TO_PRINT = 5

misses = pd.DataFrame(missed_rows)
oracle_misses = misses[misses['experiment'] == INSPECTION_EXPERIMENT].reset_index(drop=True)
experiment = next(item for item in EXPERIMENTS if item['name'] == INSPECTION_EXPERIMENT)
print(f"Inspecting {len(oracle_misses)} missed gold chunks from {INSPECTION_EXPERIMENT}.")

for miss_number, miss in oracle_misses.iterrows():
    query, ticker, gold_chunk_id = miss['question'], miss['ticker'], miss['gold_chunk_id']
    top_results = hybrid_retrieve(
        query, model, MODEL_CONFIG['query_prefix'], normalized_embeddings, bm25_retriever,
        all_chunks, RRF_K, experiment['candidate_k'], {ticker},
    )[:TOP_RETRIEVED_TO_PRINT]
    print('\n' + '=' * 120)
    print(f"MISS {miss_number + 1}/{len(oracle_misses)} | ticker={ticker} | gold={gold_chunk_id}")
    print(f"QUESTION: {query}")
    print(
        'GOLD RANKS: '
        f"dense={miss['within_company_dense_rank']} | "
        f"bm25={miss['within_company_bm25_rank']}"
    )
    print('\nGOLD CHUNK TEXT:\n' + chunks_by_id[gold_chunk_id]['text'])
    print('\nTOP 5 RETRIEVED CHUNKS:')
    for rank, result in enumerate(top_results, start=1):
        retrieved_chunk = chunks_by_id[result['chunk_id']]
        print('\n' + '-' * 100)
        print(
            f"[{rank}] {result['chunk_id']} | rrf={result['rrf_score']:.6f} | "
            f"dense_rank={result['dense_rank']} | bm25_rank={result['bm25_rank']}"
        )
        print(retrieved_chunk['text'])


Inspecting 25 missed gold chunks from oracle_company_candidate_200.



MISS 1/25 | ticker=MBLY | gold=MBLY-2025-CHUNK-000003
QUESTION: What revenue did Mobileye report for 2025, 2024, and 2023?
GOLD RANKS: dense=175 | bm25=41

GOLD CHUNK TEXT:
Item 1 — Business
Company Overview

We believe that our industry-leading technology platform, built upon over 25 years of research, development, data collection and validation, and purpose-built software and hardware design, gives us a differentiated ability to not only deliver excellent safety ratings and maintain a leadership position with our ADAS solutions, but also to make the mass deployment of autonomous driving solutions a reality. We also believe that the breadth of our solutions, combined with our global customer base, represents a significant market opportunity for us. Our platform is efficient and modular by design, enabling our customers to productize our most advanced solutions today and then leverage those investments to launch even more advanced systems in a modular and incremental manner. Our solut


MISS 2/25 | ticker=MBLY | gold=MBLY-2025-CHUNK-000003
QUESTION: What percentage increase in revenue did Mobileye report for 2025 compared with 2024?
GOLD RANKS: dense=181 | bm25=29

GOLD CHUNK TEXT:
Item 1 — Business
Company Overview

We believe that our industry-leading technology platform, built upon over 25 years of research, development, data collection and validation, and purpose-built software and hardware design, gives us a differentiated ability to not only deliver excellent safety ratings and maintain a leadership position with our ADAS solutions, but also to make the mass deployment of autonomous driving solutions a reality. We also believe that the breadth of our solutions, combined with our global customer base, represents a significant market opportunity for us. Our platform is efficient and modular by design, enabling our customers to productize our most advanced solutions today and then leverage those investments to launch even more advanced systems in a modular and inc


MISS 3/25 | ticker=MBLY | gold=MBLY-2025-CHUNK-000003
QUESTION: How does Mobileye's fiscal-year convention contextualize the annual revenue figures it reports for 2025, 2024, and 2023?
GOLD RANKS: dense=185 | bm25=32

GOLD CHUNK TEXT:
Item 1 — Business
Company Overview

We believe that our industry-leading technology platform, built upon over 25 years of research, development, data collection and validation, and purpose-built software and hardware design, gives us a differentiated ability to not only deliver excellent safety ratings and maintain a leadership position with our ADAS solutions, but also to make the mass deployment of autonomous driving solutions a reality. We also believe that the breadth of our solutions, combined with our global customer base, represents a significant market opportunity for us. Our platform is efficient and modular by design, enabling our customers to productize our most advanced solutions today and then leverage those investments to launch even more a


MISS 4/25 | ticker=MBLY | gold=MBLY-2025-CHUNK-000003
QUESTION: What Physical AI expansion did Mobileye make with the Mentee Robotics acquisition, and what revenue did Mobileye report for 2025?
GOLD RANKS: dense=122 | bm25=188

GOLD CHUNK TEXT:
Item 1 — Business
Company Overview

We believe that our industry-leading technology platform, built upon over 25 years of research, development, data collection and validation, and purpose-built software and hardware design, gives us a differentiated ability to not only deliver excellent safety ratings and maintain a leadership position with our ADAS solutions, but also to make the mass deployment of autonomous driving solutions a reality. We also believe that the breadth of our solutions, combined with our global customer base, represents a significant market opportunity for us. Our platform is efficient and modular by design, enabling our customers to productize our most advanced solutions today and then leverage those investments to launch e

...


MISS 21/25 | ticker=QCOM | gold=QCOM-2025-CHUNK-000002
QUESTION: How do Qualcomm's product and licensing revenue model, intellectual-property portfolio, and fiscal-year conventions define the business described in the filing?
GOLD RANKS: dense=95 | bm25=39

GOLD CHUNK TEXT:
Item 1 — Business
Overview

We are a global technology leader, helping to bring intelligent computing everywhere through the development and commercialization of foundational technologies, including on-device artificial intelligence (AI), high-performance and low-power computing and advanced wireless connectivity. Our platforms help power intelligent devices that people and businesses rely on every day across industries and applications from handsets to other areas, including automotive and the internet of things (IoT). In automotive, our Snapdragon® Digital Chassis™ platforms, including connectivity, digital cockpit and advanced driver assistance and automated driving (ADAS/AD), are helping to connect the car to i


MISS 22/25 | ticker=APTV | gold=APTV-2025-CHUNK-000017
QUESTION: What does Aptiv disclose about customer concentration in 2025, and which significant materials does it procure for manufacturing?
GOLD RANKS: dense=286 | bm25=280

GOLD CHUNK TEXT:
Item 1 — BUSINESS
Customers

We sell our products and services to the major global OEMs in every region of the world. Our ten largest customers accounted for approximately 56% of our total net sales for the year ended December 31, 2025, which included approximately 10% to an individual Global OEM.

TOP 5 RETRIEVED CHUNKS:

----------------------------------------------------------------------------------------------------
[1] APTV-2025-CHUNK-000492 | rrf=0.032522 | dense_rank=2 | bm25_rank=1
Item 8 — FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA
Remaining Performance Obligations

For production parts, customer contracts generally are represented by a combination of a current purchase order and a current production schedule issued by the customer


MISS 23/25 | ticker=APTV | gold=APTV-2025-CHUNK-000158
QUESTION: What were Aptiv's 2025 total net sales and gross-margin percentage, how did gross-margin percentages differ by segment, and what was each segment's adjusted operating income?
GOLD RANKS: dense=39 | bm25=93

GOLD CHUNK TEXT:
Item 7 — MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS
Results of Operations by Segment

Units: percent
Header context: Year Ended December 31,

| Line item | 2025 | 2024 |
| :--- | ---: | ---: |
| Advanced Safety and User Experience | 18.7% | 19.0% |
| Engineered Components Group | 26.0% | 25.6% |
| Electrical Distribution Systems | 12.2% | 11.7% |
| Total | 19.1% | 18.8% |

TOP 5 RETRIEVED CHUNKS:

----------------------------------------------------------------------------------------------------
[1] APTV-2025-CHUNK-000155 | rrf=0.032522 | dense_rank=2 | bm25_rank=1
Item 7 — MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS


MISS 24/25 | ticker=APTV | gold=APTV-2025-CHUNK-000160
QUESTION: What were Aptiv's 2025 total net sales and gross-margin percentage, how did gross-margin percentages differ by segment, and what was each segment's adjusted operating income?
GOLD RANKS: dense=21 | bm25=92

GOLD CHUNK TEXT:
Item 7 — MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS
Results of Operations by Segment

Units: usd_millions

| Line item | Year Ended December 31, — 2025 | Year Ended December 31, — 2024 | Year Ended December 31, — Favorable/ (unfavorable) | Variance Due To: — Volume, net of contractual price reductions | Variance Due To: — Operational performance | Variance Due To: — Other | Variance Due To: — Total |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Advanced Safety and User Experience | $658 | $714 | $(56) | $28 | $41 | $(125) | $(56) |
| Engineered Components Group | $1,129 | $1,073 | $56 | $55 | $82 | $(81) | $56 |
| Electrical Distribution System


MISS 25/25 | ticker=APTV | gold=APTV-2025-CHUNK-000193
QUESTION: What are the principal amount and maturity of Aptiv's notes maturing in September 2028, what total debt and finance-lease obligations are contractually due, and what were 2025 capital expenditures?
GOLD RANKS: dense=104 | bm25=27

GOLD CHUNK TEXT:
Item 7 — MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS
Capital Expenditures

Capital Expenditures
Units: usd_millions
Header context: Year Ended December 31,

| Line item | 2025 | 2024 | 2023 |
| :--- | ---: | ---: | ---: |
| Advanced Safety and User Experience | $157 | $201 | $207 |
| Engineered Components Group | 314 | 368 | 423 |
| Electrical Distribution Systems | 160 | 212 | 216 |
| Other (1) | 25 | 49 | 60 |
| Total capital expenditures | $656 | $830 | $906 |
| North America | $209 | $299 | $355 |
| Europe, Middle East & Africa | 235 | 295 | 288 |
| Asia Pacific | 200 | 226 | 252 |
| South America | 12 | 10 | 11 |
| Total capital ex